# DistilBERT Intent Classifier — Retraining Notebook
### Sunlytics Conversational Recommender System (CRS) — M3 Implementation

**Purpose:** Retrain the DistilBERT classifier on the combined dataset  
(all existing v2 data 10,696 + full 40,000 synthetic = 50,696 rows).

**Strategy:** Continue from `distilbert-base-uncased` (NOT from old checkpoint).  
Reason: the old model was trained on biased data — starting fresh on the combined  
dataset gives the model a clean slate to learn the corrected boundaries.

---
**Labels and retrieval strategies:**

| ID | Label | Strategy | Combined Train Count |
|----|-------|----------|----------------------|
| 0 | INITIAL_REQUEST | FULL | 6,440 |
| 1 | REFINEMENT | FULL | 7,772 |
| 2 | ATTRIBUTE_QUESTION | PARTIAL | 5,944 |
| 3 | EXPLANATION_WHY | PARTIAL | 5,880 |
| 4 | COMPARISON | PARTIAL | 5,882 |
| 5 | SELECTION_REFERENCE | PARTIAL | 5,951 |
| 6 | FEEDBACK | NO | 6,461 |
| 7 | CHITCHAT | NO | 6,366 |
| | **TOTAL TRAIN** | | **50,696** |
| | Val (unchanged) | | 1,787 |
| | Test (held-out) | | 1,787 |

**Runtime:** ~45-55 min on Colab T4 GPU for 5 epochs over 50k samples.

## STEP 1 — Verify GPU and install dependencies

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install transformers>=4.40.0 datasets scikit-learn pandas numpy matplotlib seaborn tqdm accelerate -q

## STEP 2 — Upload your data files

Upload these files from your local machine. You need:
- `v3_train_full_50k.csv`  ← the 50,696 row combined training set (download from Claude)
- `v2_val_augmented.csv`   ← your existing validation set (from your project data folder)
- `v2_test_augmented.csv`  ← your existing test set (from your project data folder)

Run the cell below to upload.

In [ ]:
from google.colab import files
import os

print('Upload: v3_train_full_50k.csv, v2_val_augmented.csv, v2_test_augmented.csv')
uploaded = files.upload()

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/best_model', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

import shutil
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'  Moved {fname} → data/{fname}')

print('Files in data/:', os.listdir('data/'))

## STEP 3 — Configuration (all hyperparameters in one place)

In [ ]:
# ============================================================
# CONFIG — match your local config.py exactly
# ============================================================

TRAIN_FILE = 'data/v3_train_full_50k.csv'
VAL_FILE   = 'data/v2_val_augmented.csv'
TEST_FILE  = 'data/v2_test_augmented.csv'

MODEL_SAVE_DIR = 'outputs/best_model'
RESULTS_DIR    = 'outputs/results'

# Start from the pretrained HuggingFace base model (NOT the old checkpoint)
# Reason: old model learned wrong boundaries for REFINEMENT vs FEEDBACK.
# Training from base on combined data gives a clean, unbiased result.
PRETRAINED_MODEL = 'distilbert-base-uncased'

LABEL_NAMES = [
    'INITIAL_REQUEST',      # 0 → FULL
    'REFINEMENT',           # 1 → FULL
    'ATTRIBUTE_QUESTION',   # 2 → PARTIAL
    'EXPLANATION_WHY',      # 3 → PARTIAL
    'COMPARISON',           # 4 → PARTIAL
    'SELECTION_REFERENCE',  # 5 → PARTIAL
    'FEEDBACK',             # 6 → NO
    'CHITCHAT',             # 7 → NO
]
NUM_LABELS = len(LABEL_NAMES)

RETRIEVAL_STRATEGY_MAP = {
    0: 'FULL', 1: 'FULL',
    2: 'PARTIAL', 3: 'PARTIAL', 4: 'PARTIAL', 5: 'PARTIAL',
    6: 'NO', 7: 'NO',
}

MAX_LEN    = 256
BATCH_SIZE = 32       # reduce to 16 if you see CUDA out-of-memory
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3
SEED = 42

print('Configuration loaded.')
print(f'  Train file : {TRAIN_FILE}')
print(f'  Val file   : {VAL_FILE}')
print(f'  Base model : {PRETRAINED_MODEL}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Epochs     : {NUM_EPOCHS}')
print(f'  LR         : {LEARNING_RATE}')

## STEP 4 — Seed and Dataset

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


class RetrievalDataset(Dataset):
    """
    Identical to your local dataset.py.
    Reads input_text + label columns from CSV.
    """
    def __init__(self, csv_path, tokenizer):
        df = pd.read_csv(csv_path)
        self.texts  = df['input_text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        assert all(0 <= l < NUM_LABELS for l in self.labels), \
            'Found label outside 0-7 range!'

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


# ── Quick dataset sanity check ────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_FILE)
df_val   = pd.read_csv(VAL_FILE)
df_test  = pd.read_csv(TEST_FILE)

print(f'\nDataset sizes:')
print(f'  Train : {len(df_train):,}')
print(f'  Val   : {len(df_val):,}')
print(f'  Test  : {len(df_test):,}')

print(f'\nTrain label distribution:')
for i, name in enumerate(LABEL_NAMES):
    n = (df_train['label'] == i).sum()
    bar = '█' * (n // 200)
    print(f'  {i} {name:<25} {n:>5}  {bar}')

## STEP 5 — Load tokenizer and model

In [ ]:
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    get_linear_schedule_with_warmup,
)

print(f'Loading tokenizer: {PRETRAINED_MODEL}')
tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL)

print(f'Loading model: {PRETRAINED_MODEL}')
model = DistilBertForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL,
    num_labels=NUM_LABELS,
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')

## STEP 6 — DataLoaders, Optimiser, Scheduler

In [ ]:
train_dataset = RetrievalDataset(TRAIN_FILE, tokenizer)
val_dataset   = RetrievalDataset(VAL_FILE,   tokenizer)

# num_workers=0 in Colab to avoid multiprocessing issues
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=0, pin_memory=True,
)

# AdamW with no weight decay on bias / LayerNorm (standard BERT fine-tuning)
no_decay = ['bias', 'LayerNorm.weight']
optimiser = torch.optim.AdamW(
    [
        {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': WEIGHT_DECAY},
        {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ],
    lr=LEARNING_RATE,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimiser,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Steps per epoch : {len(train_loader)}')
print(f'Total steps     : {total_steps}')
print(f'Warmup steps    : {warmup_steps}')

## STEP 7 — Training loop with early stopping

In [ ]:
import json
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score, classification_report


def train_one_epoch(model, loader, optimiser, scheduler, device):
    model.train()
    total_loss = 0.0
    bar = tqdm(loader, desc='  Training', leave=False)
    for batch in bar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimiser.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()
        bar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  Evaluating', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return (total_loss / len(loader),
            f1_score(all_labels, all_preds, average='macro'),
            all_preds, all_labels)


# ── Training loop ─────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
best_val_f1 = 0.0
epochs_no_improve = 0

print('=' * 55)
print('Starting training on combined dataset (v3)...')
print('=' * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    print('-' * 40)

    train_loss = train_one_epoch(model, train_loader, optimiser, scheduler, device)
    val_loss, val_f1, val_preds, val_labels = evaluate(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_f1)

    print(f'  Train loss   : {train_loss:.4f}')
    print(f'  Val loss     : {val_loss:.4f}')
    print(f'  Val macro-F1 : {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        model.save_pretrained(MODEL_SAVE_DIR)
        tokenizer.save_pretrained(MODEL_SAVE_DIR)
        print(f'  ✓ New best model saved  (val macro-F1 = {best_val_f1:.4f})')
        report = classification_report(val_labels, val_preds,
                                       target_names=LABEL_NAMES, digits=4)
        with open(f'{RESULTS_DIR}/best_epoch_val_report.txt', 'w') as f:
            f.write(f'Best epoch: {epoch}\n\n{report}')
    else:
        epochs_no_improve += 1
        print(f'  No improvement. Patience: {epochs_no_improve}/{EARLY_STOPPING_PATIENCE}')

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f'\nEarly stopping after epoch {epoch}.')
        break

with open(f'{RESULTS_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Best val macro-F1: {best_val_f1:.4f}')

## STEP 8 — Test set evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Load BEST saved model for test eval
print('Loading best model for test evaluation...')
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
best_tokenizer = DistilBertTokenizer.from_pretrained(MODEL_SAVE_DIR)
best_model     = DistilBertForSequenceClassification.from_pretrained(MODEL_SAVE_DIR)
best_model.to(device)
best_model.eval()

test_dataset = RetrievalDataset(TEST_FILE, best_tokenizer)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2,
                           shuffle=False, num_workers=0, pin_memory=True)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy    = accuracy_score(all_labels, all_preds)
macro_f1    = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
per_class   = f1_score(all_labels, all_preds, average=None)

print('\n' + '='*55)
print('TEST SET RESULTS')
print('='*55)
print(f'  Accuracy    : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Macro-F1    : {macro_f1:.4f}')
print(f'  Weighted-F1 : {weighted_f1:.4f}')
print('\n  Per-class F1:')
for name, f1 in zip(LABEL_NAMES, per_class):
    print(f'    {name:<25} {f1:.4f}')

report = classification_report(all_labels, all_preds,
                                target_names=LABEL_NAMES, digits=4)
print('\nClassification Report:')
print(report)

with open(f'{RESULTS_DIR}/test_classification_report.txt', 'w') as f:
    f.write('TEST SET CLASSIFICATION REPORT\n')
    f.write('='*55 + '\n\n')
    f.write(report)

metrics = {
    'accuracy': round(float(accuracy), 4),
    'macro_f1': round(float(macro_f1), 4),
    'weighted_f1': round(float(weighted_f1), 4),
    'per_class_f1': {n: round(float(f), 4) for n, f in zip(LABEL_NAMES, per_class)},
}
with open(f'{RESULTS_DIR}/test_metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics JSON saved.')

## STEP 9 — Confusion matrix and training curves

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — v3 Retrained Model (row-normalised)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('CrossEntropy Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, history['val_macro_f1'], 'g-o', label='Val Macro-F1')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('Validation Macro-F1 per Epoch')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('DistilBERT CRS Classifier — v3 Combined Training', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## STEP 10 — Quick inference demo (verify the model fixes the failures)

In [ ]:
def predict_single(history_turns, current_msg):
    parts = [f"{t['role'].upper()}: {t['content']}" for t in history_turns[-6:]]
    parts.append(f'CURRENT: {current_msg}')
    input_text = ' [SEP] '.join(parts)

    enc = best_tokenizer(
        input_text, truncation=True, padding='max_length',
        max_length=MAX_LEN, return_tensors='pt'
    )
    with torch.no_grad():
        logits = best_model(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device)
        ).logits
    probs   = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    label_id = int(probs.argmax())
    return {
        'label': LABEL_NAMES[label_id],
        'strategy': RETRIEVAL_STRATEGY_MAP[label_id],
        'confidence': f'{probs[label_id]*100:.1f}%',
    }


# Shared history for cases 2-4
hist = [
    {'role': 'user', 'content': 'I need a dress for a party'},
    {'role': 'bot',  'content': 'Option 1 is the Spring dress (black, dress). Option 2 is the Carnival dress (red, dress).'},
]

tests = [
    # (description,  history,  message,  expected)
    ('INITIAL_REQUEST',      [],   'I need a casual top for work',              'INITIAL_REQUEST'),
    ('REFINEMENT (cheap)',   hist, 'need a cheaper one',                         'REFINEMENT'),
    ('REFINEMENT (cheaper?)',hist, 'cheaper?',                                   'REFINEMENT'),
    ('EXPLANATION_WHY short',hist, 'why?',                                       'EXPLANATION_WHY'),
    ('EXPLANATION_WHY frag', hist, 'why this one',                               'EXPLANATION_WHY'),
    ('FEEDBACK pos',         hist, "I'll take it",                               'FEEDBACK'),
    ('FEEDBACK neg price',   hist, 'too expensive',                              'FEEDBACK'),
    ('CHITCHAT',             [],   'hello',                                      'CHITCHAT'),
    ('COMPARISON',           hist, 'which one is better quality?',               'COMPARISON'),
    ('SELECTION_REFERENCE',  hist, 'tell me more about the second one',          'SELECTION_REFERENCE'),
    ('ATTRIBUTE_QUESTION',   hist, 'is the first one machine washable?',         'ATTRIBUTE_QUESTION'),
]

print(f'{"Test":<28} {"Predicted":<22} {"Expected":<22} {"OK?":<5} {"Conf"}')
print('-' * 90)
correct = 0
for desc, h, msg, expected in tests:
    r = predict_single(h, msg)
    ok = '✅' if r['label'] == expected else '❌'
    if r['label'] == expected: correct += 1
    print(f'{desc:<28} {r["label"]:<22} {expected:<22} {ok}  {r["confidence"]}')

print(f'\nPassed: {correct}/{len(tests)}')

## STEP 11 — Download the trained model

This zips the `outputs/` folder and downloads it so you can copy the `best_model/` back into your local `outputs/` directory.

In [ ]:
import shutil
from google.colab import files

# Zip the entire outputs folder
shutil.make_archive('crs_v3_outputs', 'zip', 'outputs')
print('Zipped outputs → crs_v3_outputs.zip')
print('Downloading...')
files.download('crs_v3_outputs.zip')
print('Download started.')

## After downloading

1. Extract `crs_v3_outputs.zip`
2. Copy `best_model/` folder into your local `outputs/` directory (replacing the old one)
3. Your `predict.py` and `pipeline.py` will automatically use the new model because they read from `MODEL_SAVE_DIR = outputs/best_model`
4. Run your local `python evaluate.py` to regenerate the full test report with plots

**Expected improvements vs the old model:**
- `"need a cheaper one"` → REFINEMENT (was misclassified as FEEDBACK)
- `"cheaper?"`           → REFINEMENT
- `"why?"` / `"why this?"` → EXPLANATION_WHY (was CHITCHAT)
- `"too expensive"` (no follow-up ask) → FEEDBACK (boundary now sharp)